In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.day12_customers(
    CustomerId INT,
    CustomerName STRING,
    City STRING,
    Age INT,
    UpdatedAt TIMESTAMP
)
USING DELTA;

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

customer_schema = StructType([
    StructField("CustomerId", IntegerType(), True),
    StructField("CustomerName", StringType(), True),
    StructField("City", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("UpdatedAt", StringType(), True)
])
bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("rescuedDataColumn", "_rescued_data")
    .schema(customer_schema)
    .option("cloudFiles.schemaLocation", "/Volumes/workspace/bronze/schema/day12_customers/")
    .load("/Volumes/workspace/bronze/customer_data_files/")
)

In [0]:
def process_batch(batch_df, batch_id):
    batch_df.write.mode("append").saveAsTable("bronze.day12_customers")
query = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("checkpointLocation", "/Volumes/workspace/bronze/checkpoints/day12_customer_data/")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [0]:
%sql
select * from bronze.day12_customers order by customerid;

CustomerId,CustomerName,City,Age,UpdatedAt,_rescued_data
101,Arun,Bangalore,31,2026-08-21 10:00:00,null
101,Arun,Chennai,30,2026-08-20 09:00:00,null
102,Kumar,Hyderabad,36,2026-08-22 09:00:00,null
102,Kumar,Bangalore,35,2026-08-20 09:05:00,null
103,Priya,Chennai,27,2026-08-20 09:10:00,null
104,Ravi,Coimbatore,32,2026-08-21 10:05:00,null
105,Meena,Madurai,29,2026-08-22 09:05:00,null
107,RajKumar,Hyderabad,27,2026-08-22 09:00:00,null
108,Ravi,Madurai,27,2026-08-22 09:05:00,null


In [0]:
def process_silver(batch_df, batch_id):

    from pyspark.sql.functions import col, to_timestamp, row_number
    from pyspark.sql.window import Window
    from delta.tables import DeltaTable

    batch_df = batch_df.withColumn(
        "UpdatedAt",
        to_timestamp(col("UpdatedAt"))
    )

    window_spec = (
        Window
        .partitionBy("CustomerId")
        .orderBy(col("UpdatedAt").desc())
    )

    latest_batch = (
        batch_df
        .withColumn("row_num", row_number().over(window_spec))
        .filter(col("row_num") == 1)
        .drop("row_num")
        .drop("rescued_data")
    )

    silver_table = DeltaTable.forName(
        spark,
        "silver.day12_customers"
    )

    (
        silver_table.alias("target")
        .merge(
            latest_batch.alias("source"),
            "target.CustomerId = source.CustomerId"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

query = (
    bronze_stream.writeStream
    .foreachBatch(process_silver)
    .option("checkpointLocation", "/Volumes/workspace/silver/checkpoints/day12_silver/")
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

In [0]:
%sql
select * from silver.day12_customers;

CustomerId,CustomerName,City,Age,UpdatedAt
101,Arun,Bangalore,31,2026-08-21T10:00:00.000Z
102,Kumar,Hyderabad,36,2026-08-22T09:00:00.000Z
103,Priya,Chennai,27,2026-08-20T09:10:00.000Z
104,Ravi,Coimbatore,32,2026-08-21T10:05:00.000Z
105,Meena,Madurai,29,2026-08-22T09:05:00.000Z
107,RajKumar,Hyderabad,27,2026-08-22T09:00:00.000Z
108,Ravi,Madurai,27,2026-08-22T09:05:00.000Z
